In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, upper
from pyspark.sql.types import DecimalType

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
df = spark.read.parquet("/user/hive/warehouse/sclsepb4532_raw.db/products", header=True, inferSchema=True)

26/11/06 08:22:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 08:22:47 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/11/06 08:24:48 WARN DataStreamer: Slow waitForAckedSeqno took 57573ms (threshold=30000ms). File being written: /user/sclsepb4532/.sparkStaging/application_1793087731018_0310/__spark_libs__1500682139869919055.zip, block: BP-1364978691-172.31.64.10-1677754292516:blk_1101584108_27850830, Write pipeline datanodes: [DatanodeInfoWithStorage[172.31.64.13:9866,DS-e595a26d-2fb4-4683-b8bf-ad5a6b69a54b,DISK], DatanodeInfoWithStorage[172.31.64.14:9866,DS-f823cf84-06f8-4cdd-b6e9-29d13d0292ee,DISK], DatanodeInfoWithStorage[172.31.64.11:9866,DS-8af7bd92-c7a4-44fb-b361-00496b8456c3,DISK]].


In [4]:
df_cleaned = df.select(
    trim(col("product_id")).alias("product_id"),
    trim(col("product_name")).alias("product_name"),
    trim(col("category")).alias("category"),
    trim(col("sub_category")).alias("sub_category"),
    trim(col("brand")).alias("brand"),
    col("price").cast(DecimalType(12, 2)).alias("price"),
    col("cost_price").cast(DecimalType(12, 2)).alias("cost_price"),
    to_date(trim(col("launch_date")), "yyyy-MM-dd").alias("launch_date"),
    upper(trim(col("is_active"))).alias("is_active")
)

df_staging = df_cleaned.filter(
    col("product_id").isNotNull() &
    (col("product_id") != "") &
    col("product_name").isNotNull() &
    (col("product_name") != "") &
    col("price").isNotNull() &
    (col("price") >= 0) &
    col("cost_price").isNotNull() &
    (col("cost_price") >= 0) &
    col("launch_date").isNotNull() &
    col("is_active").isin("Y", "N")
)

In [5]:
df_staging.write \
    .mode("overwrite") \
    .saveAsTable("sclsepb4532_staging.products_stg")